# 🎯 Calibrate Violations - General

---

## 📊 Overview

This notebook documents how we  **calibrating and validating violation parameters** in synthetic datasets.


### 📝 Purpose

To ensure that generated violations are calibrated in a manner that cover the full degeneration range (based on AUROC (1-0.5)) for the baseline strategy, we explored the exact violation steps that we use at the end here.

In [ ]:
from calibration_tools import check_violation_range
import numpy as np
from hydra import initialize, compose

# $V_{conf,l}$

In [22]:
fail_rate = []
with initialize(version_base=None, config_path="../../config/"):
    cfg = compose(config_name="generate_dataset.yaml")
performance_stack = []
for x in [0, 0.135, 0.27625, 0.4175, 0.55875, 0.7]:
    cfg.generator.lagged.alternativ_parameter_range = [0.8, 0.9]
    cfg.generator.remove_n_variables_for_confounding = 1
    cfg.generator.lagged.alternative_link_proba = x
    cfg.generator.lagged.alternative_coeff_ts = 1
    res, fails = check_violation_range(cfg)
    performance_stack.append(res)
    fail_rate.append(fails)
print(np.array(performance_stack).round(5))
print(np.array(fail_rate).sum())


[0.96964 0.91781 0.73872 0.64503 0.63949 0.52923]
0


# $V_{conf,i}$

In [23]:
# We use the same setup for consistency and as it never fully degenerates the lagged effect estimation
with initialize(version_base=None, config_path="../../config/"):
    cfg = compose(config_name="generate_dataset.yaml")
performance_stack = []
fail_rate = []
for x in [0, 0.135, 0.27625, 0.4175, 0.55875, 0.7]:
    cfg.generator.exog.link_proba = x
    cfg.generator.instant.link_proba = 0.1
    cfg.generator.exog.param_range = [0.8, 0.9]
    res, fails = check_violation_range(cfg)
    performance_stack.append(res)
    fail_rate.append(fails)
print(np.array(performance_stack).round(5))
print(np.array(fail_rate).sum())

[0.86997 0.86712 0.87044 0.86424 0.85959 0.85603]
0


# $V_{faith,l}$

In [25]:
# Not much to calibrate here, as the mask is never removing all links. # Check the generate masks notebook for details.

paths = [
    None,
    "../masks/masks_5_4_faith_lagged_4",
    "../masks/masks_5_4_faith_lagged_3",
    "../masks/masks_5_4_faith_lagged_2",
    "../masks/masks_5_4_faith_lagged_1",
    "../masks/masks_5_4_faith_lagged_0",
]

with initialize(version_base=None, config_path="../../config/"):
    cfg = compose(config_name="generate_dataset.yaml")
performance_stack = []
fail_rate = []
for x in paths:
    cfg.generator.lagged.link_proba = 0 if x else 0.075
    cfg.link_mask_path = x
    res, fails = check_violation_range(cfg)
    performance_stack.append(res)
    fail_rate.append(fails)
print(np.array(performance_stack).round(5))
print(np.array(fail_rate).sum())

[0.9975  0.86639 0.8373  0.80261 0.78033 0.77208]
0


# $V_{faith,i}$

In [31]:
# Same levels used here. SHould affect the lagged estimation

paths = [
    None,
    "../masks/masks_5_faith_instant_4",
    "../masks/masks_5_faith_instant_3",
    "../masks/masks_5_faith_instant_2",
    "../masks/masks_5_faith_instant_1",
    "../masks/masks_5_faith_instant_0",
]

with initialize(version_base=None, config_path="../../config/"):
    cfg = compose(config_name="generate_dataset.yaml")
performance_stack = []
fail_rate = []

for x in paths:
    cfg.instant_link_mask_path = x
    cfg.generator.instant.link_proba = 0
    res, fails = check_violation_range(cfg)
    performance_stack.append(res)
    fail_rate.append(fails)
print(np.array(performance_stack).round(5))
print(np.array(fail_rate).sum())

[0.93912 0.93912 0.93912 0.93912 0.93912 0.93912]
0


# $V_{faith,z}$

In [80]:
levels=[
    [0.3,0.5],
    [0.12,0.24],
    [0.0975,0.1925],
    [0.075,0.145],
    [0.0525,0.0975],
    [0.03,0.05]
]

with initialize(version_base=None, config_path="../../config/"):
    cfg = compose(config_name="generate_dataset.yaml")
performance_stack = []
fail_rate = []

for x in levels:
    cfg.generator.lagged.param_range = x
    cfg.generator.instant.param_range = x
    res, fails = check_violation_range(cfg)
    performance_stack.append(res)
    fail_rate.append(fails)
print(np.array(performance_stack).round(5))
print(np.array(fail_rate).sum())

[0.93912 0.90699 0.84318 0.75239 0.62136 0.4994 ]
0


# $V_{empty}$

In [90]:
with initialize(version_base=None, config_path="../../config/"):
    cfg = compose(config_name="generate_dataset.yaml")
performance_stack = []
fail_rate = []
for x in [
    [250],
    [40, 100, 150, 210],
    [31, 106, 144, 219],
    [23, 111, 139, 227],
    [14, 117, 133, 236],
    [6, 122, 128, 244],
]:
    cfg.nonstationary_change = 0
    cfg.generator.nonstationary = True
    cfg.generator.drop_struc_for_window = True
    cfg.generator.change_points = x
    res, fails = check_violation_range(cfg)
    performance_stack.append(res)
    fail_rate.append(fails)
print(np.array(performance_stack).round(5))
print(np.array(fail_rate).sum())

[0.93912 0.90329 0.86583 0.79911 0.67314 0.50643]
0


# $V_{length}$

In [113]:
with initialize(version_base=None, config_path="../../config/"):
    cfg = compose(config_name="generate_dataset.yaml")
performance_stack = []
fail_rate = []

for x in [
    250,
    76,
    58,
    41,
    23,
    6,
]:  # lets not go down to 5 to make it a single sample. this will break methods.
    cfg.generator.time_series_n = x
    res, fails = check_violation_range(cfg)
    performance_stack.append(res)
    fail_rate.append(fails)
print(np.array(performance_stack).round(5))
print(np.array(fail_rate).sum())


[0.93912 0.90413 0.89289 0.83637 0.76602 0.54123]
0


# $V_{MCAR}$

In [126]:
with initialize(version_base=None, config_path="../../config/"):
    cfg = compose(config_name="generate_dataset.yaml")
performance_stack = []
fail_rate = []

for x in [
    0,
    0.2,
    0.3375,
    0.475,
    0.6125,
    0.75,
]:  # lets not go down to 5 to make it a single sample. this will break methods.
    cfg.generator.interpolate = x
    res, fails = check_violation_range(cfg)
    performance_stack.append(res)
    fail_rate.append(fails)
print(np.array(performance_stack).round(5))
print(np.array(fail_rate).sum())


[0.93912 0.89981 0.77443 0.64817 0.60134 0.52128]
0


# $V_{MAR}$

In [128]:
# Keep the same setup as before for consistency between the missingness types as they are technically comparable.
with initialize(version_base=None, config_path="../../config/"):
    cfg = compose(config_name="generate_dataset.yaml")
performance_stack = []
fail_rate = []
for x in [
    0,
    0.2,
    0.3375,
    0.475,
    0.6125,
    0.75,
]:  # lets not go down to 5 to make it a single sample. this will break methods.
    cfg.generator.interpolate = x
    cfg.generator.missingness_type = "MAR"
    cfg.generator.missingness_base_path = "../semi_synthetic_bases/rivers_ts_flood.csv"
    res, fails = check_violation_range(cfg)
    performance_stack.append(res)
    fail_rate.append(fails)
print(np.array(performance_stack).round(5))
print(np.array(fail_rate).sum())

[0.93912 0.88803 0.76913 0.6712  0.56155 0.53516]
0


# $V_{MNAR}$

In [131]:
# Keep the same setup as before for consistency.
with initialize(version_base=None, config_path="../../config/"):
    cfg = compose(config_name="generate_dataset.yaml")
performance_stack = []
fail_rate = []

for x in [
    0,
    0.2,
    0.3375,
    0.475,
    0.6125,
    0.75,
]:  # lets not go down to 5 to make it a single sample. this will break methods.
    cfg.generator.interpolate = x
    cfg.generator.missingness_type = "MNAR"
    cfg.generator.missingness_base_path = "../semi_synthetic_bases/rivers_ts_flood.csv"
    res, fails = check_violation_range(cfg)
    performance_stack.append(res)
    fail_rate.append(fails)
print(np.array(performance_stack).round(5))
print(np.array(fail_rate).sum())

[0.93912 0.76773 0.67084 0.60187 0.57019 0.53824]
0


# $V_{Scale}$

In [132]:
# This should do nothing as we test it for the Notears-based methods.
with initialize(version_base=None, config_path="../../config/"):
    cfg = compose(config_name="generate_dataset.yaml")
performance_stack = []
for x in [0, 0.2, 0.4, 0.6, 0.8, 1]:
    cfg.generator.standardization_factor = x
    cfg.generator.normalize = "standardization"
    res, fails = check_violation_range(cfg)
    performance_stack.append(res)
    fail_rate.append(fails)
print(np.array(performance_stack).round(5))
print(np.array(fail_rate).sum())

[0.93912 0.93912 0.93912 0.93912 0.93912 0.93912]
0


## $V_{coef}$
 

In [162]:
# This is a little tricky
# if we change higher than 1 we potentially jump out of the specified parameter range likey causing divergence
# This range has some interesting properties as it can push parameters small for some periods.
with initialize(version_base=None, config_path="../../config/"):
    cfg = compose(config_name="generate_dataset.yaml")
performance_stack = []
fail_rate = []
for x in [0, 0.275, 0.33125, 0.3875, 0.44375, 0.5]:
    cfg.generator.change_points = [50, 100, 150, 200]
    cfg.generator.nonstationary = True
    cfg.nonstationary_change = x
    res, fails = check_violation_range(cfg)
    performance_stack.append(res)
    fail_rate.append(fails)
print(np.array(performance_stack).round(5))
print(np.array(fail_rate).sum())

[0.93912 0.90875 0.85178 0.80777 0.76418 0.74525]
0


## $V_{stat}$

In [174]:
# Problematic violation  as it cannot be properly scaled.
# Note for small windows (e.g. [20]), the effects are really strong as the smaller regime cannot be correctly estimated.

with initialize(version_base=None, config_path="../../config/"):
    cfg = compose(config_name="generate_dataset.yaml")
performance_stack = []
fail_rate = []
change_points = [
    [],
    [125],
    [83, 166],
    [63, 126, 187],
    [50, 100, 150, 200],
    [41, 82, 122, 163, 205],
]
for x in change_points:
    cfg.generator.change_points = x
    cfg.generator.nonstationary = False
    res, fails = check_violation_range(cfg)
    performance_stack.append(res)
    fail_rate.append(fails)
print(np.array(performance_stack).round(5))
print(np.array(fail_rate).sum())

[0.93912 0.78985 0.68283 0.64052 0.61474 0.60165]
0
